# Packages  import

In [34]:
import os
import json
import  requests
from bs4 import BeautifulSoup

# Ceneo scraper

1. Provide url address of product's opoinions webpage

In [ ]:
product_code = "124893467"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"

2. Send the request to provided url address

In [25]:
response = requests.get(url)
print(response.status_code)

200


3. If status code is OK, fetch product name

In [26]:
page_dom = BeautifulSoup(response.text, 'html.parser')
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [27]:
product_name = page_dom.select_one("h1.product-top__product-info__name").get_text(strip=True)
print(product_name)

Urządzenie wielofunkcyjne HP Smart Tank 670 AiO (6UU48A)


4. If status code is OK, fetch all opinions from requested webpage

In [ ]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")

print(type(opinions))
print(len(opinions))


<class 'bs4.element.ResultSet'>
10


5. For all fetched opinions, parse them to extract relevant data

In [32]:
all_opinions = []

for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion['data-entry-id'],
        'author': opinion.select_one("span.user-post__author-name").get_text().strip(),
        'recommendation': opinion.select_one("span.user-post__author-recommendation > em").get_text().strip() if opinion.select_one("span.user-post__author-recommendation > em") else None,
        'score': opinion.select_one("span.user-post__score-count").get_text().strip(),
        'content': opinion.select_one("div.user-post__text").get_text().strip(),
        'pros': [p.get_text().strip() for p in opinion.select_one("div.review-feature__item--positive")] if opinion.select_one("div.review-feature__item--positive") else [],
        'cons': [c.get_text().strip() for c in opinion.select_one("div.review-feature__item--negative")] if opinion.select_one("div.review-feature__item--negative") else [],
        'likes': opinion.select_one("button.vote-yes > span").get_text().strip(),
        'dislikes': opinion.select_one("button.vote-no > span").get_text().strip(),
        'publication_date': opinion.select_one("span.user-post__published > time:nth-child(1)")['datetime'].strip(),
        'purchase_date': opinion.select_one("span.user-post__published > time:nth-child(2)")['datetime'].strip() if opinion.select_one("span.user-post__published > time:nth-child(2)[datetime]") else None
    }
    all_opinions.append(single_opinion)

6. Check if next page with opinions

In [ ]:
next = True if page_dom.select_one("a.pagination__next") else False
if next: page += 1

8. Save obtained opinions

In [35]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [36]:
with open(f"./opinions/{product_code}.json", "w", encoding="UTF-8") as json_file:
    json.dump(all_opinions, json_file, indent=4, ensure_ascii=False)